# GenAI Assignment – Mastering Prompt Templates in LangChain
**Internship:** Data Science Internship – February 2026 | Innomatics Research Labs  
**Topic:** Building dynamic and reusable prompt systems using LangChain instead of hardcoded strings

---

### Why Prompt Templates?
Hardcoded f-strings break the moment you need to reuse a prompt with different inputs.  
LangChain's `PromptTemplate` and `ChatPromptTemplate` solve this by separating the structure from the values.

### Pipeline
```
User Input → Input Validation → Prompt Template → Formatted Prompt → Output
```

---
## Setup — Install and Import

In [1]:
# Install LangChain core packages
!pip install -q langchain langchain-core
print("Packages ready.")

Packages ready.


In [2]:
# Import prompt classes from LangChain
# PromptTemplate     → for single-turn, text-based prompts
# ChatPromptTemplate → for multi-turn conversation models (system + human messages)
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

print("Imports successful.")

Imports successful.


---
## Task 1: Replace Hardcoded Prompts with Templates

Instead of writing `f"Explain {topic} in simple terms"` every time,  
define it once as a template and reuse it with any input.

In [3]:
# Define a reusable template with one variable: {subject}
beginner_explainer = PromptTemplate(
    input_variables=["subject"],
    template="Explain {subject} in simple terms suitable for a beginner"
)

# Test with a single input
sample_output = beginner_explainer.format(subject="Machine Learning")
print("Single test:")
print(sample_output)

Single test:
Explain Machine Learning in simple terms suitable for a beginner


In [4]:
# Reuse the same template across multiple subjects
# This is the key benefit — one definition, infinite reuse
subjects_list = ["Cybersecurity", "Internet of Things", "Natural Language Processing"]

print("Reusing template with different inputs:\n")
for subj in subjects_list:
    formatted = beginner_explainer.format(subject=subj)
    print(formatted)
    print()

Reusing template with different inputs:

Explain Cybersecurity in simple terms suitable for a beginner

Explain Internet of Things in simple terms suitable for a beginner

Explain Natural Language Processing in simple terms suitable for a beginner



---
## Task 2: Multi-Input Prompt System

Templates can accept multiple variables at once.  
Here we use three: `subject`, `target_group`, and `style`.

In [5]:
# Three-variable template: adapts explanation based on subject, audience, and tone
adaptive_prompt = PromptTemplate(
    input_variables=["subject", "target_group", "style"],
    template="Explain {subject} to {target_group} using a {style} communication style"
)

# Sample test scenarios
scenarios = [
    {"subject": "Generative AI",   "target_group": "high school students", "style": "casual"},
    {"subject": "SQL Databases",   "target_group": "business analysts",    "style": "professional"},
    {"subject": "Reinforcement Learning", "target_group": "ML engineers", "style": "technical"}
]

print("Multi-input prompt results:\n")
for idx, scenario in enumerate(scenarios, start=1):
    output = adaptive_prompt.format(**scenario)
    print(f"Scenario {idx}: {output}")
    print()

Multi-input prompt results:

Scenario 1: Explain Generative AI to high school students using a casual communication style

Scenario 2: Explain SQL Databases to business analysts using a professional communication style

Scenario 3: Explain Reinforcement Learning to ML engineers using a technical communication style



---
## Task 3: Prompt Variations Engine

Same topic, three different prompt styles — teaching, quiz, and narrative.  
Each style is its own template object.

In [6]:
# Three distinct prompt styles for the same input topic

step_by_step_prompt = PromptTemplate(
    input_variables=["subject"],
    template="Break down {subject} into clear, numbered steps for a learner"
)

quiz_prompt = PromptTemplate(
    input_variables=["subject"],
    template="Generate 3 conceptual quiz questions to test understanding of {subject}"
)

narrative_prompt = PromptTemplate(
    input_variables=["subject"],
    template="Tell the concept of {subject} as a short engaging story"
)

# Map style names to their template objects
prompt_styles = {
    "Step-by-Step": step_by_step_prompt,
    "Quiz"        : quiz_prompt,
    "Narrative"   : narrative_prompt
}

chosen_subject = "Deep Learning"

print(f"Prompt variations for topic: '{chosen_subject}'\n")
print("-" * 50)
for style_name, tmpl in prompt_styles.items():
    print(f"{style_name}:")
    print(f"  {tmpl.format(subject=chosen_subject)}")
    print()

Prompt variations for topic: 'Deep Learning'

--------------------------------------------------
Step-by-Step:
  Break down Deep Learning into clear, numbered steps for a learner

Quiz:
  Generate 3 conceptual quiz questions to test understanding of Deep Learning

Narrative:
  Tell the concept of Deep Learning as a short engaging story



---
## Task 4: ChatPromptTemplate with Role-Based System Messages

`ChatPromptTemplate` supports system + human message pairs.  
The system message sets the model's behavior; the human message is the actual question.

In [7]:
# System messages that define how the assistant should behave
assistant_personas = {
    "mentor"    : "You are a patient mentor who uses real-world analogies to explain technical topics.",
    "examiner"  : "You are a rigorous examiner who probes deep understanding through challenging questions.",
    "coach"     : "You are an energetic coach who keeps learners motivated and focused on practical outcomes."
}

def create_role_prompt(persona_key, subject):
    """
    Build a ChatPromptTemplate for a given persona and subject.

    Args:
        persona_key (str): One of 'mentor', 'examiner', 'coach'
        subject (str): The topic to discuss

    Returns:
        list: Formatted message objects (SystemMessage + HumanMessage)
    """
    # Get system message for the chosen persona
    system_instruction = assistant_personas.get(
        persona_key,
        "You are a helpful assistant."
    )

    # Build the chat template with system + human turn
    role_template = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        ("human", "Can you help me understand {subject}?")
    ])

    return role_template.format_messages(subject=subject)


# Test all three personas on the same subject
test_subject = "Transformer Architecture"
personas_to_test = ["mentor", "examiner", "coach"]

print(f"ChatPromptTemplate — Role-based prompts for: '{test_subject}'\n")
print("=" * 55)

for persona in personas_to_test:
    msg_list = create_role_prompt(persona, test_subject)
    print(f"Persona: {persona}")
    for msg in msg_list:
        msg_type = type(msg).__name__
        print(f"  [{msg_type}]: {msg.content}")
    print()

ChatPromptTemplate — Role-based prompts for: 'Transformer Architecture'

Persona: mentor
  [SystemMessage]: You are a patient mentor who uses real-world analogies to explain technical topics.
  [HumanMessage]: Can you help me understand Transformer Architecture?

Persona: examiner
  [SystemMessage]: You are a rigorous examiner who probes deep understanding through challenging questions.
  [HumanMessage]: Can you help me understand Transformer Architecture?

Persona: coach
  [SystemMessage]: You are an energetic coach who keeps learners motivated and focused on practical outcomes.
  [HumanMessage]: Can you help me understand Transformer Architecture?



---
## Task 5: Input Validation Layer

Before sending inputs to a template, validate them.  
Unknown values get replaced with safe defaults — no crashes, no bad prompts.

In [8]:
# Define allowed values for each parameter
ALLOWED_TARGET_GROUPS = ["beginner", "intermediate", "expert"]
ALLOWED_STYLES        = ["formal", "casual", "fun"]

def sanitize_inputs(target_group, style):
    """
    Validate target_group and style against allowed values.
    Falls back to safe defaults if an invalid value is passed.

    Args:
        target_group (str): Intended audience for the prompt
        style (str): Communication style for the prompt

    Returns:
        tuple: (validated_target_group, validated_style)
    """
    # Validate target group
    if target_group not in ALLOWED_TARGET_GROUPS:
        print(f"  [Warning] '{target_group}' is not recognised. Falling back to 'beginner'.")
        target_group = "beginner"

    # Validate style
    if style not in ALLOWED_STYLES:
        print(f"  [Warning] '{style}' is not a valid style. Falling back to 'formal'.")
        style = "formal"

    return target_group, style


# Validation test cases
validation_tests = [
    ("beginner",  "fun"),       # Both valid
    ("newcomer",  "casual"),    # Invalid target_group
    ("expert",    "serious"),   # Invalid style
    ("advanced",  "angry"),     # Both invalid
]

for i, (tg, st) in enumerate(validation_tests, start=1):
    print(f"Test {i} — Input: target_group='{tg}', style='{st}'")
    result = sanitize_inputs(tg, st)
    print(f"  Output: {result}")
    print()

Test 1 — Input: target_group='beginner', style='fun'
  Output: ('beginner', 'fun')

Test 2 — Input: target_group='newcomer', style='casual'
  [Warning] 'newcomer' is not recognised. Falling back to 'beginner'.
  Output: ('beginner', 'casual')

Test 3 — Input: target_group='expert', style='serious'
  [Warning] 'serious' is not a valid style. Falling back to 'formal'.
  Output: ('expert', 'formal')

Test 4 — Input: target_group='advanced', style='angry'
  [Warning] 'advanced' is not recognised. Falling back to 'beginner'.
  [Warning] 'angry' is not a valid style. Falling back to 'formal'.
  Output: ('beginner', 'formal')



---
## Task 6: Prompt Generator Application

Combines validation + style selection + template formatting into one function.  
Pass any topic, audience, tone, and style — get a clean prompt back.

In [9]:
# Style-to-template mapping — each style has a unique prompt structure
prompt_library = {
    "teaching": PromptTemplate(
        input_variables=["subject", "target_group", "style"],
        template="Teach {subject} to {target_group} using a {style} approach"
    ),
    "interview": PromptTemplate(
        input_variables=["subject", "target_group", "style"],
        template="Prepare 3 interview questions on {subject} for a {target_group} candidate, asked in a {style} manner"
    ),
    "storytelling": PromptTemplate(
        input_variables=["subject", "target_group", "style"],
        template="Narrate {subject} as a story written for {target_group} in a {style} tone"
    )
}

DEFAULT_PROMPT_STYLE = "teaching"  # Fallback when an unknown style is requested

def build_prompt(subject, target_group, style, prompt_style):
    """
    Generate a validated, formatted prompt string.

    Args:
        subject (str)      : Topic to cover
        target_group (str) : Intended audience
        style (str)        : Communication tone
        prompt_style (str) : Which prompt template to use

    Returns:
        str: The formatted prompt ready for an LLM
    """
    # Step 1: Validate audience and tone
    target_group, style = sanitize_inputs(target_group, style)

    # Step 2: Resolve prompt style, fall back if unknown
    if prompt_style not in prompt_library:
        print(f"  [Warning] '{prompt_style}' is not a known style. Using '{DEFAULT_PROMPT_STYLE}'.")
        prompt_style = DEFAULT_PROMPT_STYLE

    # Step 3: Format and return the prompt
    selected_template = prompt_library[prompt_style]
    return selected_template.format(
        subject=subject,
        target_group=target_group,
        style=style
    )


# Test the generator with different inputs
generator_tests = [
    ("Large Language Models",  "beginner",     "fun",    "storytelling"),
    ("SQL Optimization",       "expert",       "formal", "interview"),
    ("Data Visualization",     "intermediate", "casual", "teaching"),
]

print("Prompt Generator Output:\n")
print("-" * 55)
for subj, tg, st, ps in generator_tests:
    print(f"Input  → subject='{subj}', target='{tg}', style='{st}', mode='{ps}'")
    prompt_output = build_prompt(subj, tg, st, ps)
    print(f"Output → {prompt_output}")
    print()

Prompt Generator Output:

-------------------------------------------------------
Input  → subject='Large Language Models', target='beginner', style='fun', mode='storytelling'
Output → Narrate Large Language Models as a story written for beginner in a fun tone

Input  → subject='SQL Optimization', target='expert', style='formal', mode='interview'
Output → Prepare 3 interview questions on SQL Optimization for a expert candidate, asked in a formal manner

Input  → subject='Data Visualization', target='intermediate', style='casual', mode='teaching'
Output → Teach Data Visualization to intermediate using a casual approach



---
## Task 7: Template Reusability Test

One template definition.  
Five completely different input sets.  
Proves the core value of prompt templates over hardcoded strings.

In [10]:
# Single template used across all five tests
universal_template = PromptTemplate(
    input_variables=["subject", "target_group", "style"],
    template="Explain {subject} to a {target_group} using a {style} tone"
)

# Five diverse input combinations
batch_inputs = [
    {"subject": "Gradient Descent",       "target_group": "beginner",     "style": "casual"},
    {"subject": "Kubernetes",              "target_group": "expert",       "style": "formal"},
    {"subject": "ETL Pipelines",           "target_group": "intermediate", "style": "fun"},
    {"subject": "Feature Engineering",    "target_group": "beginner",     "style": "formal"},
    {"subject": "Graph Neural Networks",   "target_group": "expert",       "style": "casual"},
]

print("Reusability test — same template, 5 different inputs:\n")
print("=" * 55)
for num, inp in enumerate(batch_inputs, start=1):
    generated_prompt = universal_template.format(**inp)
    print(f"Run {num}: {generated_prompt}")
    print()

Reusability test — same template, 5 different inputs:

Run 1: Explain Gradient Descent to a beginner using a casual tone

Run 2: Explain Kubernetes to a expert using a formal tone

Run 3: Explain ETL Pipelines to a intermediate using a fun tone

Run 4: Explain Feature Engineering to a beginner using a formal tone

Run 5: Explain Graph Neural Networks to a expert using a casual tone



---
## Full Pipeline Demo

End-to-end run: raw user input → validation → template selection → final prompt.  
This simulates how a real application would use the system.

In [11]:
print("=" * 55)
print("FULL PIPELINE DEMO")
print("=" * 55)

# Simulated raw user input (would come from a form or API in production)
raw_subject      = "Convolutional Neural Networks"
raw_target_group = "newbie"       # Invalid — will trigger fallback
raw_style        = "aggressive"   # Invalid — will trigger fallback
raw_mode         = "storytelling" # Valid

print("Step 0 — Raw Input Received:")
print(f"  subject      : {raw_subject}")
print(f"  target_group : {raw_target_group}")
print(f"  style        : {raw_style}")
print(f"  mode         : {raw_mode}")
print()

# Step 1: Validate inputs
print("Step 1 — Validation:")
clean_target, clean_style = sanitize_inputs(raw_target_group, raw_style)
print(f"  target_group after validation : {clean_target}")
print(f"  style after validation        : {clean_style}")
print()

# Step 2: Generate final prompt
print("Step 2 — Prompt Generation:")
final_output = build_prompt(raw_subject, raw_target_group, raw_style, raw_mode)
print(f"  Final Prompt : {final_output}")
print()
print("Pipeline complete.")
print("=" * 55)

FULL PIPELINE DEMO
Step 0 — Raw Input Received:
  subject      : Convolutional Neural Networks
  target_group : newbie
  style        : aggressive
  mode         : storytelling

Step 1 — Validation:
  [Warning] 'newbie' is not recognised. Falling back to 'beginner'.
  [Warning] 'aggressive' is not a valid style. Falling back to 'formal'.
  target_group after validation : beginner
  style after validation        : formal

Step 2 — Prompt Generation:
  [Warning] 'newbie' is not recognised. Falling back to 'beginner'.
  [Warning] 'aggressive' is not a valid style. Falling back to 'formal'.
  Final Prompt : Narrate Convolutional Neural Networks as a story written for beginner in a formal tone

Pipeline complete.


---
## Summary

| Task | What Was Built |
|---|---|
| Task 1 | Replaced hardcoded f-strings with reusable `PromptTemplate` |
| Task 2 | Multi-variable template (subject + target_group + style) |
| Task 3 | Three prompt style variants for the same topic |
| Task 4 | `ChatPromptTemplate` with role-based system messages |
| Task 5 | Input validation with safe fallback defaults |
| Task 6 | Combined prompt generator with validation + style routing |
| Task 7 | Reusability test — one template across five input sets |

### Key Takeaway
Prompt Templates separate **structure** from **content**.  
Once a template is defined, the logic never changes — only the inputs do.  
This is the same principle behind good function design: write once, reuse everywhere.

---
*Completed as part of GenAI Internship – February 2026 | Innomatics Research Labs*  
*Portfolio: siddharthkaulagi.vercel.app*